# 01 · Water detection tests

Runs Contract 5 (`pipeline/utils/sar.py`) on a preprocessed NISAR scene with several thresholds. This is how the L-band tuning in `docs/methodology.md` was found: the dry lake bed is nearly as dark as water in HH, so Otsu runs inside the search zone and is capped at -18 dB HH / -23 dB HV.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pipeline.utils import sar, geo

In [ ]:
f = sorted((ROOT / 'data/interim/nisar_pre').glob('*/*.npz'))[0]
d = np.load(f)
hh, hv, search, valid, res = d['hh_f'], d['hv_f'], d['search'], d['valid'], d['reservoir']
hand = d['hand_ok'] if d['hand_ok'].shape == hh.shape else None
f

In [ ]:
z = search & valid
plt.hist(hh[z], bins=100, range=(-35, -5), alpha=.6, label='HH in search zone')
plt.hist(hv[z], bins=100, range=(-35, -5), alpha=.6, label='HV in search zone')
plt.legend();
print('Otsu HH:', sar.otsu_threshold(hh[z], -18, (-30, -18)), ' HV:', sar.otsu_threshold(hv[z], -23, (-35, -23)))

In [ ]:
rows = []
for thr in [-14, -16, -17, -18, -19]:
    for hvt in [None, -22, -23, -24]:
        w = (hh < thr) & search & valid
        if hand is not None: w &= hand
        if hvt: w &= hv < hvt
        w = sar.cleanup(w, 3, 10, res)
        rows.append({'hh_thr': thr, 'hv_thr': hvt, 'area_km2': round(w.sum() * 100 / 1e6, 1)})
pd.DataFrame(rows).pivot(index='hh_thr', columns='hv_thr', values='area_km2')

In [ ]:
p = sar.DetectParams.from_cfg(__import__('yaml').safe_load(open(ROOT/'config/settings.yaml'))['detection'], 'nisar')
from dataclasses import replace
p = replace(p, median_px=1)
r = sar.detect_water(hh, valid, search, p, reservoir=res, hand_ok=hand, cross_db=hv)
plt.figure(figsize=(10, 5)); plt.imshow(hh, cmap='gray', vmin=-28, vmax=0)
plt.imshow(np.ma.masked_where(~r.water, r.water), cmap='Blues', alpha=.6)
plt.title(f'{r.method} threshold {r.threshold_db:.1f} dB, area {r.water.sum()*100/1e6:.1f} km2'); plt.axis('off');